# 04b. Multivariable Logistic Regression


Short summary:
[04b - Multivariable Outcomes Analysis (Short Summary)](../docs/04b_multivariable_outcomes_SHORT.md)

## 1. Purpose

This analysis evaluates the association between early exposure to renin–angiotensin–aldosterone system (RAAS) inhibitors and in-hospital mortality among adult **non-ICU hospital admissions** derived from the MIMIC-IV v3.1 database.

Models adjusted for age, gender, race group, admission type, insurance, and anchor year group.

A retrospective cohort was constructed at the hospital-admission level (one row per admission), excluding any admissions with documented ICU stays. Documented early RAAS prescription exposure was defined from records with recorded start times from admission through less than 24 hours; it does not confirm administration. The primary outcome was in-hospital mortality, as indicated by the hospital_expire_flag.

To account for measured confounding, multivariable logistic regression models were fitted, adjusting for age, gender, race group, admission type, insurance, and anchor year group. Model results are presented using both conventional odds ratios and **average marginal effects (AMEs)** to improve interpretability on the absolute risk scale.

In addition, age-specific risk differences were estimated to illustrate contrasts at a typical covariate profile across age. No exposure-by-age interaction was fitted, so these curves are not a formal test of effect modification. Statistical uncertainty was quantified using individual-level bootstrap resampling, with the full modeling pipeline repeated to construct 95% confidence intervals.

Use the generated OR, risk and uncertainty summaries to describe this execution’s associations. Marginal effects and age-specific risk contrasts complement odds ratios, but none establishes a causal treatment effect or an exposure-by-age interaction.

## 2. Background

Renin–angiotensin–aldosterone system (RAAS) inhibitors, including angiotensin-converting enzyme inhibitors (ACE inhibitors) and angiotensin receptor blockers (ARBs), are widely prescribed for cardiovascular and renal diseases. Beyond blood pressure control, RAAS inhibition has been hypothesized to confer protective effects through modulation of inflammation, endothelial function, and organ perfusion.

In critically ill and hospitalized populations, however, the impact of early RAAS inhibitor exposure on short-term mortality remains uncertain. Clinical decision-making is complicated by concerns regarding hemodynamic instability, acute kidney injury, and indication bias, leading to heterogeneous prescribing practices across institutions and patient subgroups.

Previous observational studies have often relied on odds ratios derived from multivariable logistic regression to assess associations between medication exposure and mortality. While odds ratios are statistically convenient, they are frequently misinterpreted as risk measures and can exaggerate effect sizes when outcomes are not rare. This limitation reduces their usefulness for clinical interpretation and decision support.

To address these challenges, this study focuses on marginal effect–based estimates, including average marginal effects (AME) and age-specific risk differences (RD), which directly quantify absolute risk changes. Using a large, real-world hospital dataset (MIMIC-IV), we apply a transparent and reproducible analytical pipeline combining multivariable adjustment with bootstrap-based uncertainty estimation.

The objective of this project is not to establish causal effects, but to demonstrate how regression-based models can be translated into clinically interpretable risk measures, and how uncertainty can be rigorously quantified in observational clinical data.

## 3. Methods

### 3.1 Data Source and Study Population

- **MIMIC-IV v3.1** (BigQuery public dataset)
- Project: `mimic-iv-portfolio`

**Source Tables:**
  - `mimic-iv-portfolio.nonicu_raas.analysis_dataset`<br>
    (created in [02_exposure.ipynb](02_exposure.ipynb) using [03_build_analysis_dataset.sql](../sql/03_build_analysis_dataset.sql))

This retrospective observational study used data from the Medical Information Mart for Intensive Care IV (MIMIC-IV), a large, publicly available, de-identified clinical database containing detailed electronic health record data from hospital admissions at Beth Israel Deaconess Medical Center.

The study population consisted of adult patients (aged ≥ 18 years) with at least one recorded hospital admission. Admissions with missing key identifiers or implausible demographic values were excluded. Each hospital admission was treated as an independent observation. Patient age at admission was derived from anchor age information provided by MIMIC-IV.

Because the database is fully de-identified, institutional review board approval and informed consent were not required.

### 3.2 Exposure and Outcome Definitions

The primary exposure of interest was early use of renin–angiotensin–aldosterone system (RAAS) inhibitors, defined using inpatient medication prescription records indicating initiation of either an angiotensin-converting enzyme inhibitor (ACE inhibitor) or an angiotensin receptor blocker (ARB) during the early phase of hospitalization.

Exposure was operationalized as a binary indicator representing any early RAAS inhibitor use. Additional binary indicators distinguishing ACE inhibitor and ARB use were retained for descriptive analyses and sensitivity checks.

The primary outcome was in-hospital mortality, defined using the hospital expiration flag in MIMIC-IV. Mortality during the index hospitalization was coded as 1, and survival to discharge as 0.

### 3.3 Covariates

Multivariable adjustment was performed using a predefined set of demographic, admission-related, insurance, and calendar-time covariates selected a priori based on clinical relevance and data availability. These included:

- Age at admission (continuous)
- Sex
- Race and ethnicity
- Admission type
- Insurance category
- Calendar period of admission (anchor year group)

Categorical variables were encoded using one-hot encoding, with reference categories omitted to avoid perfect multicollinearity. Missing values in categorical covariates were explicitly coded as an “Unknown” category prior to encoding.

### 3.4 Statistical Analysis

A multivariable logistic regression model was fitted to estimate the association between early RAAS inhibitor exposure and in-hospital mortality, adjusting for the covariates described above.

Three original fits have distinct roles: class-balanced sklearn logistic regression is used only for discrimination; unpenalized statsmodels Logit produced the historical OR diagnostics and is nonconverged under separation; an unweighted L2-regularized sklearn model supplies predicted risks and AME. The explicitly added Firth sensitivity fits supply finite adjusted ORs with separate uncertainty analyses. These fitted objects are not interchangeable.

### 3.5 Marginal Effects and Risk Differences

To improve clinical interpretability beyond odds ratios, model results were summarized using marginal effect–based measures derived from predicted probabilities.

The primary estimand was the average marginal effect (AME), defined as the average absolute difference in predicted in-hospital mortality risk between exposed and unexposed admissions.

In addition, age-specific risk differences (RDs) were estimated by computing adjusted predicted mortality probabilities across a grid of ages, comparing exposure groups at each age.

Average predicted risks and AME average two exposure scenarios over admissions with their observed covariates. Age-specific curves instead fix non-age covariates at their full-cohort means/proportions and vary age; this typical-profile calculation is distinct from marginal standardization over the observed population.

### 3.6 Bootstrap-Based Uncertainty Estimation

Statistical uncertainty was quantified using a nonparametric bootstrap procedure. Individual hospital admissions were resampled with replacement, and the full analysis pipeline—including feature encoding, model fitting, and marginal effect estimation—was repeated for each bootstrap sample.

The original 200-replicate admission-level bootstrap computes percentile intervals only for age-specific typical-profile risk differences; it did not compute an AME confidence interval. The added patient-cluster procedure separately refits models and computes percentile intervals for Firth ORs, AME, average predicted risks and age-specific contrasts. It resamples patients with all their eligible admissions, and reports fitting failures. The original admission-random discrimination split is a historical reference; the added patient-disjoint holdout prevents patient overlap.


**Discrimination interpretation:** The patient-disjoint holdout is an internal evaluation of retrospective discrimination on the full admission cohort. It uses prescription records from 0–<24 hours while including deaths within 24 hours, and applies no 24-hour survival/continued-hospitalization eligibility filter. There is no fixed clinical prediction time or prospective prediction horizon: The generated retrospective ROC AUC and PR AUC do not describe admission-time or 24-hour prognostic performance. Patient separation and train-only preprocessing address patient overlap and preprocessing leakage, not temporal feature availability. The 24-hour landmark association analyses separately require confirmed survival and continued hospitalization strictly beyond 24 hours and evaluate subsequent in-hospital death; they use a different eligible cohort.


### 3.7 Effect Measures and Definitions

Effect measures used in this study are defined as follows:

- **Odds Ratio (OR)**  
  The model-based odds ratio derived from the logistic regression coefficient
  for early RAAS exposure (binary indicator).
  This OR represents the relative change in odds of in-hospital mortality
  associated with early RAAS exposure, conditional on covariates included
  in the model.  
  ORs are reported for reference only.

- **Average Marginal Effect (AME)**  
  The average difference in predicted mortality risk when early RAAS exposure
  is set to 1 versus 0, averaged over all individuals in the study population.
  AME was defined as:

  $$
  \mathrm{AME}
  = \frac{1}{N} \sum_{i=1}^{N}
  \Big[
  P(Y = 1 \mid \mathrm{RAAS} = 1, X_i)
  -
  P(Y = 1 \mid \mathrm{RAAS} = 0, X_i)
  \Big]
  $$

  where:
  - **Outcome (Y)** is a binary indicator of the primary outcome.
    - Y = 1 indicates occurrence of the outcome (e.g., in-hospital mortality).
    - Y = 0 indicates no occurrence.
  - **Exposure (RAAS)** indicates early RAAS inhibitor use
    (RAAS = 1 for early use, 0 otherwise).

  AME was used as the primary effect measure due to its direct clinical interpretability.

- **Risk Difference by Age (RD(age))**  
  The age-specific absolute risk difference in predicted mortality between
  early RAAS exposure and no exposure.
  RD(age) was estimated at a typical covariate profile by fixing non-age
  covariates at their sample means and varying age across a predefined grid.

## 4. Modeling Data Preparation
### 4.1 Raw Analysis Dataset Loading

## Reading the results

Data sources and execution mode are summarized once below, after the destination settings. Expand **Calculation inputs and source definitions** to trace individual SQL definitions and aggregate references.

Each result is generated by Python in this order: **number/title → figure or aggregate table → how to read → computed result summary → interpretation limits**. Cohort definitions, exposures, adjustment variables and estimators are unchanged. See the [reproduction instructions](../REPRODUCIBILITY.md#python-generated-notebook-results).


The input-definition index describes the Python objects/queries used in calculation. The separate **Saved aggregates and verification files (CSV/JSON)** list identifies saved results and checking records, rather than original MIMIC-IV source tables. Live notebook displays use in-memory results; saved-output replay and validation workflows may read these files.

**Live input behavior:** Normal execution requires the configured real BigQuery inputs and approved access. Missing settings, credentials, permissions or tables stop execution with a cause and a next step; no synthetic/demo data or saved results are substituted. Stored presentation-replay outputs are review artifacts, not a successful new data load. Use a fresh kernel for a new run.


In [ ]:
# Presentation uses only objects produced by this notebook execution.
from pathlib import Path as _ResultPath
import sys as _result_sys
_result_root = _ResultPath.cwd().resolve()
if _result_root.name == "notebooks":
    _result_root = _result_root.parent
if not (_result_root / "scripts" / "nonicu_notebook_results.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory")
if str(_result_root / "scripts") not in _result_sys.path:
    _result_sys.path.insert(0, str(_result_root / "scripts"))
from nonicu_notebook_results import NotebookResults
notebook_results = NotebookResults(globals())


In [ ]:
import pandas as pd
import numpy as np
import warnings

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import ConvergenceWarning

from sklearn.base import clone
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

import statsmodels.api as sm

In [ ]:
from real_data_access import real_data_access, real_data_io
from google.cloud import bigquery
from google.auth import default

# Project / dataset
PROJECT_ID = "mimic-iv-portfolio"
DATASET = "nonicu_raas"
TABLE_ANALYSIS = f"{PROJECT_ID}.{DATASET}.analysis_dataset"

# ADC credentials
with real_data_access(PROJECT_ID, DATASET):
    creds, adc_project = default()
    client = bigquery.Client(project=PROJECT_ID, credentials=creds)

@real_data_io
def query_to_df(sql: str) -> pd.DataFrame:
    # Run a SELECT query in BigQuery and return a DataFrame
    return client.query(sql).to_dataframe(create_bqstorage_client=False)

print("Connected to:", PROJECT_ID, "| ADC default:", adc_project)
print("TABLE_ANALYSIS:", TABLE_ANALYSIS)

In [ ]:
# Shared source references and execution mode for all numbered results.
notebook_results.overview(['Table 04b.1', 'Table 04b.2', 'Table 04b.3', 'Table 04b.4', 'Table 04b.5', 'Table 04b.6', 'Table 04b.7', 'Figure 04b.D1', 'Table 04b.8', 'Table 04b.9', 'Figure 04b.D2', 'Table 04b.10', 'Figure 04b.1', 'Figure 04b.2', 'Table 04b.11', 'Table 04b.12', 'Figure 04b.3', 'Table 04b.13', 'Table 04b.14', 'Table 04b.15', 'Figure 04b.4', 'Figure 04b.5'])


In [ ]:
with notebook_results.capture('Table 04b.1', globals()):
    required_landmark_fields = {"admittime", "dischtime", "deathtime", "hospital_expire_flag"}
    analysis_schema = {field.name: field.field_type for field in client.get_table(TABLE_ANALYSIS).schema}
    missing_landmark_fields = required_landmark_fields.difference(analysis_schema)
    if missing_landmark_fields:
        raise ValueError(f"Analysis table lacks required landmark fields: {sorted(missing_landmark_fields)}")
    print("Landmark field types:", {c: analysis_schema[c] for c in sorted(required_landmark_fields)})
    q = f"SELECT * FROM `{TABLE_ANALYSIS}`"
    df = query_to_df(q)

    print("rows:", df.shape[0], "cols:", df.shape[1])
    _notebook_result_value = (df.info())


### 4.2 Outcome and Exposure Construction

In [ ]:
# Outcome and main exposure (keep ONE exposure variable)
df = df.copy()
df["outcome"] = df["hospital_expire_flag"].astype(int)
df["exposure"] = df["raas_any_early"].astype(int)   # RAAS early (any ACEi/ARB)

# Feature engineering
num_vars = ["age"] 
cat_vars = ["gender", "race_group", "admission_type", "insurance", "anchor_year_group"]
age_grid = np.arange(18, 95)
df = df.drop(columns=["race"], errors="ignore")

### 4.3 Feature Engineering

In [ ]:
# Handle missing values in categorical variables
for c in cat_vars:
    df[c] = df[c].fillna("Unknown")

# One-hot encoding
df_ohe = pd.get_dummies(df, columns=cat_vars, drop_first=True)

### 4.4 Final Modeling Matrix Construction

In [ ]:
with notebook_results.capture('Table 04b.2', globals()):
    # Keep only modeling columns (avoid leakage / unnecessary columns)
    feature_cols = ["exposure"] + num_vars \
        + [c for c in df_ohe.columns if c.startswith("gender_")] \
        + [c for c in df_ohe.columns if c.startswith("race_group_")] \
        + [c for c in df_ohe.columns if c.startswith("admission_type_")] \
        + [c for c in df_ohe.columns if c.startswith("insurance_")] \
        + [c for c in df_ohe.columns if c.startswith("anchor_year_group_")]

    forbidden_cols = {"hosp_los", "dischtime", "deathtime", "hospital_expire_flag", "outcome"}
    bad_in_vars = sorted(forbidden_cols.intersection(num_vars + cat_vars))
    bad_in_features = sorted(forbidden_cols.intersection(feature_cols))
    assert not bad_in_vars, f"forbidden cols in num_vars/cat_vars: {bad_in_vars}"
    assert not bad_in_features, f"forbidden cols in feature_cols: {bad_in_features}"

    X = df_ohe[feature_cols].copy()
    y = df_ohe["outcome"].copy()

    _notebook_result_value = (print("X shape:", X.shape, "Outcome rate:", y.mean()))


## 5. Outputs
### 5.1 Study Population and Baseline Characteristics

In [ ]:
with notebook_results.capture('Table 04b.3', globals()):
    # Descriptive statistics
    # Overall sample size
    df.shape

    # Age distribution
    df["age"].describe()

    # Early RAAS exposure: counts and proportions
    raas_counts = df["raas_any_early"].value_counts()
    raas_props  = df["raas_any_early"].value_counts(normalize=True)

    pd.DataFrame({
        "n": raas_counts,
        "proportion": raas_props
    })

    # Unadjusted in-hospital mortality by early RAAS exposure
    mortality_summary = df.groupby("raas_any_early")["hospital_expire_flag"].agg(
        n_admissions="count",
        n_events="sum",
        mortality_rate="mean"
    )

    _notebook_result_value = (mortality_summary)


### 5.2 Historical Retrospective Discrimination Diagnostics

In [ ]:
with notebook_results.capture('Table 04b.4', globals()):
    # 1) split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 2) scale only numeric columns (OHE dummies stay as-is)
    num_cols = num_vars  # e.g., ["age"] or other continuous vars

    preprocess = ColumnTransformer(
        transformers=[("num", StandardScaler(), num_cols)],
        remainder="passthrough"
    )

    # 3) Classification model: class weighting is used only for discrimination
    classification_model = Pipeline(steps=[
        ("prep", preprocess),
        ("logit", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
            C=1.0
        ))
    ])

    classification_model.fit(X_train, y_train)

    # 4) evaluation
    p_test = classification_model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, p_test)
    ap  = average_precision_score(y_test, p_test)

    print(f"AUC: {auc:.4f}")
    print(f"Average precision (PR-AUC): {ap:.4f}")

    # 5) Risk-estimation model: unweighted, fit on the full cohort
    # This model is the sole source for model-predicted risks and risk differences.
    def make_unweighted_risk_model():
        return Pipeline([
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                class_weight=None
            ))
        ])

    risk_model = make_unweighted_risk_model()
    risk_model.fit(X, y)
    risk_prob = risk_model.predict_proba(X)[:, 1]
    print(f"Observed mortality prevalence: {y.mean():.4%}")
    _notebook_result_value = (print(f"Mean unweighted-model predicted risk: {risk_prob.mean():.4%}"))


**Historical admission-row metric provenance**

Older execution snapshots used an unordered admission-row split with possible patient overlap. Their numerical provenance is retained in the [audit record](../docs/AUDIT_CORRECTIONS_20261118.md#discrimination-metric-provenance), rather than copied into current results. The legacy code below reports values from its current execution; the later deterministic reference and patient-disjoint holdout are separate evaluations.

The legacy category encoding precedes splitting; the later pipelines fit category encoding and scaling on training data. Both evaluate retrospective discrimination, including early deaths, without a fixed clinical prediction time. Class-balanced probabilities are classification scores, not calibrated mortality risks. They do not validate the separate Firth or unweighted risk/AME estimators.


### 5.3 Classification-Model Coefficient Diagnostic

In [ ]:
with notebook_results.capture('Table 04b.5', globals()):
    # Inspect the fitted class-weighted classifier; these coefficients are not
    # used for calibrated risk estimation or the primary exposure estimate.
    classification_logit = classification_model.named_steps["logit"]
    coef_vec = classification_logit.coef_.ravel()

    # Get feature names after preprocessing
    prep = classification_model.named_steps["prep"]
    feature_names = prep.get_feature_names_out()  # this should work in recent sklearn

    # Build coef/OR table
    coef = pd.Series(coef_vec, index=feature_names, name="coef")
    or_ = np.exp(coef).rename("OR")

    res = pd.concat([coef, or_], axis=1).sort_values("OR")

    # Stable diagnostic: show only the exposure coefficient from the classifier.
    _notebook_result_value = (res.loc[res.index.astype(str).str.contains("exposure", case=False, regex=False)])


**Classification diagnostic interpretation**

The exposure coefficient shown above belongs to the class-balanced L2 discrimination model. It is a classifier diagnostic, not the primary exposure estimate or a calibrated risk contrast. Current supplementary OR inference uses Firth fits and patient-cluster intervals; absolute risks use the separate unweighted full-cohort L2 model. The patient-disjoint holdout below is the current discrimination check.


### 5.4 Odds Ratio for Early RAAS Exposure (Multivariable Logistic Regression)

In [ ]:
with notebook_results.capture('Table 04b.6', globals()):
    # Statsmodels logistic regression for OR + 95% CI
    X_sm = sm.add_constant(X.astype(float), has_constant="add")
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Maximum Likelihood optimization failed to converge.*",
            category=ConvergenceWarning,
        )
        model = sm.Logit(y, X_sm).fit(disp=False)

    print("Historical primary MLE converged:", bool(model.mle_retvals.get("converged", False)))
    if not model.mle_retvals.get("converged", False):
        print("Diagnostic output only: do not interpret the nominal CI/p-value as validated inference; see Firth supplement.")

    params = model.params
    conf = model.conf_int()
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="overflow encountered in exp",
            category=RuntimeWarning,
        )
        or_ci = pd.DataFrame({
            "OR": np.exp(params),
            "CI_lower": np.exp(conf[0]),
            "CI_upper": np.exp(conf[1]),
            "p_value": model.pvalues
        })

    _notebook_result_value = (or_ci.loc[["exposure"]].sort_values("OR"))


**Aggregate Python Export**

Primary unpenalized parameters are exported to `python/outputs/mle_diagnostics.csv`. This nonconverged fit is retained for diagnostics only; its nominal intervals and p-values are not validated inference. The model specification is unchanged.


In [ ]:
with notebook_results.capture('Table 04b.7', globals()):
    # Export primary logistic regression parameters for downstream validation
    from pathlib import Path

    notebooks_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
    project_root = notebooks_dir.parent
    output_path = project_root / "python" / "outputs" / "mle_diagnostics.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    primary_logit_results = model
    primary_params = primary_logit_results.params
    primary_conf = primary_logit_results.conf_int()

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="overflow encountered in exp",
            category=RuntimeWarning,
        )
        python_logistic_parameters = pd.DataFrame({
            "term": primary_params.index,
            "coefficient": primary_params.values,
            "coefficient_ci_lower": primary_conf[0].values,
            "coefficient_ci_upper": primary_conf[1].values,
            "odds_ratio": np.exp(primary_params).values,
            "odds_ratio_ci_lower": np.exp(primary_conf[0]).values,
            "odds_ratio_ci_upper": np.exp(primary_conf[1]).values,
            "p_value": primary_logit_results.pvalues.values,
        })

    python_logistic_parameters.to_csv(output_path, index=False)

    _notebook_result_value = (python_logistic_parameters.loc[
        python_logistic_parameters["term"].astype(str).str.contains("exposure", case=False, regex=False)
    ])


**README Figure Export**

The historical primary MLE figure is refreshed only if the fit converges. Current README/report figures use explicitly labeled supplementary Firth estimates with patient-cluster intervals; the model specification is unchanged.


In [ ]:
with notebook_results.diagnostic('Figure 04b.D1', globals()):
    # Retain historical primary MLE figure when the model has no finite converged solution.
    if not model.mle_retvals.get("converged", False):
        print("Historical primary OR figure not refreshed: primary MLE did not converge.")
    else:
        # Save README figure: adjusted odds ratio for early RAAS exposure
        import matplotlib.pyplot as plt

        assets_dir = project_root / "assets"
        assets_dir.mkdir(parents=True, exist_ok=True)
        figure_path = assets_dir / "adjusted_odds_ratio_plot.png"

        exposure_row = python_logistic_parameters.loc[
            python_logistic_parameters["term"].eq("exposure")
        ].iloc[0]
        or_value = exposure_row["odds_ratio"]
        ci_low = exposure_row["odds_ratio_ci_lower"]
        ci_high = exposure_row["odds_ratio_ci_upper"]

        fig, ax = plt.subplots(figsize=(7.5, 2.2))
        ax.errorbar(
            or_value,
            0,
            xerr=[[or_value - ci_low], [ci_high - or_value]],
            fmt="o",
            color="#2563eb",
            ecolor="#2563eb",
            elinewidth=2.4,
            capsize=6,
            markersize=8,
        )
        ax.axvline(1.0, color="#64748b", linestyle="--", linewidth=1.5)
        ax.set_xscale("log")
        ax.set_xlim(0.2, 1.4)
        ax.set_ylim(-0.5, 0.5)
        ax.set_yticks([0])
        ax.set_yticklabels(["Early RAAS exposure"])
        ax.set_xlabel("Adjusted odds ratio for in-hospital mortality (log scale)")
        ax.set_title("Adjusted odds ratio for early RAAS exposure")
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.tick_params(axis="y", length=0)
        fig.subplots_adjust(left=0.28, right=0.96, top=0.82, bottom=0.30)
        fig.savefig(figure_path, dpi=300)
        plt.show()

        figure_path.relative_to(project_root)


### 5.5 24h Landmark Sensitivity Analysis

Eligibility requires confirmed survival **and continued hospitalization strictly beyond admission + 24 hours**. Discharge or death exactly at 24 hours is excluded. Survivors must have `hospital_expire_flag == 0`, no death time, and a valid discharge after the landmark. Deaths must have `hospital_expire_flag == 1` and `landmark < deathtime <= dischtime`; missing/invalid or contradictory timestamps and flags are excluded with explicit reasons. A missing death time is expected for a consistent live discharge, but does not establish survival for a recorded hospital death.

The outcome is subsequent in-hospital mortality among eligible admissions. The primary analysis, prescription exposure at 0–<24 hours, entire-stay non-ICU cohort, and adjustment variables are unchanged. This remains a selected observational sensitivity cohort and does not eliminate residual confounding or all immortal-time concerns.

The overlapping diagnostic table counts issues in the full input and among admissions retained by the original condition. The exclusion table assigns the first applicable reason in the listed order, so its counts sum to the input total. Only aggregates are displayed. The original and corrected conditions are refitted with both existing covariate specifications; numerical covariate exclusions, deaths and convergence are shown separately for each fit.

Numerical counts and model status are generated below. The [historical landmark validation](../docs/24H_LANDMARK_VALIDATION.md) remains an execution record rather than the source of current result values.


In [ ]:
def classify_landmark(data):
    """Return row-level decisions locally; display/export only aggregate tables."""
    required = {"admittime", "dischtime", "deathtime", "hospital_expire_flag"}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"Missing landmark fields: {sorted(missing)}")
    times = {c: pd.to_datetime(data[c], errors="coerce", utc=True)
             for c in ("admittime", "dischtime", "deathtime")}
    admit, discharge, death = (times[c] for c in ("admittime", "dischtime", "deathtime"))
    landmark = admit + pd.Timedelta(hours=24)
    flag = pd.to_numeric(data["hospital_expire_flag"], errors="coerce")
    # Reproduce the original condition; missing death time alone allowed inclusion.
    original = (data["deathtime"].isna() | ((death - admit) > pd.Timedelta(hours=24))).fillna(False)
    issues = {
        "missing_admittime": data["admittime"].isna(),
        "invalid_admittime": data["admittime"].notna() & admit.isna(),
        "missing_dischtime": data["dischtime"].isna(),
        "invalid_dischtime": data["dischtime"].notna() & discharge.isna(),
        "discharge_before_admission": discharge < admit,
        "invalid_or_missing_expire_flag": ~flag.isin([0, 1]),
        "invalid_deathtime": data["deathtime"].notna() & death.isna(),
        "survival_flag_with_deathtime": flag.eq(0) & death.notna(),
        "death_flag_without_deathtime": flag.eq(1) & death.isna(),
        "death_before_admission": death < admit,
        "death_after_discharge": death > discharge,
        "death_at_or_before_24h": death <= landmark,
        "discharge_at_or_before_24h": discharge <= landmark,
    }
    # First applicable reason wins. Diagnostics below retain overlapping issues.
    reason = pd.Series("included", index=data.index, dtype="object")
    for name, mask in issues.items():
        reason.loc[reason.eq("included") & mask.fillna(False)] = name
    eligible = reason.eq("included")
    assert (discharge.loc[eligible] > landmark.loc[eligible]).all()
    assert (flag.loc[eligible].eq(0) | (death.loc[eligible] > landmark.loc[eligible])).all()
    live_discharge = flag.eq(0) & data["deathtime"].isna() & discharge.ge(admit)
    audit = {
        **issues,
        "live_discharge_before_24h": live_discharge & (discharge < landmark),
        "live_discharge_exactly_24h": live_discharge & (discharge == landmark),
        "live_discharge_at_or_before_24h": live_discharge & (discharge <= landmark),
        "death_exactly_24h": death == landmark,
        "discharge_exactly_24h": discharge == landmark,
        "missing_deathtime_all": data["deathtime"].isna(),
    }
    diagnostics = pd.DataFrame([
        {"issue": name, "all_admissions": int(mask.fillna(False).sum()),
         "retained_by_original": int((mask.fillna(False) & original).sum())}
        for name, mask in audit.items()
    ])
    flow = pd.DataFrame({"reason": reason, "original": original, "death": flag.eq(1), "unknown_flag": ~flag.isin([0, 1])})
    exclusion_counts = flow.groupby("reason", sort=False).agg(
        admissions=("reason", "size"), flagged_deaths=("death", "sum"),
        unknown_expire_flag=("unknown_flag", "sum"),
        retained_by_original=("original", "sum")).reset_index()
    assert exclusion_counts["admissions"].sum() == len(data)
    assert not (eligible & ~original).any()
    return original, eligible, reason, diagnostics, exclusion_counts


In [ ]:
with notebook_results.capture('Table 04b.8', globals()):
    original_lm_mask, corrected_lm_mask, landmark_reason, landmark_diagnostics, landmark_exclusions = classify_landmark(df)
    display(landmark_diagnostics)  # overlapping diagnostic counts
    display(landmark_exclusions)  # mutually exclusive reasons, including eligible admissions
    print(f"Original condition: {int(original_lm_mask.sum()):,}; corrected: {int(corrected_lm_mask.sum()):,}")
    _notebook_result_value = (print("Boundary tests: run scripts/test_landmark_sensitivity.py (synthetic data only)."))


In [ ]:
with notebook_results.capture('Table 04b.9', globals()):
    from statsmodels.tools.sm_exceptions import PerfectSeparationError

    # Model specification and missing-covariate rules match the previous sensitivity cell.
    def fit_landmark_model(frame, label, with_proxy=False):
        work = frame.copy()
        categories = cat_vars + (["admission_location"] if with_proxy else [])
        for c in categories:
            work[c] = work[c].fillna("Unknown")
        encoded = pd.get_dummies(work, columns=categories, drop_first=True).dropna(subset=num_vars)
        columns = ["exposure"] + num_vars
        for c in categories:
            columns += [v for v in encoded.columns if v.startswith(f"{c}_")]
        features = encoded[columns].astype(float)
        outcome = pd.to_numeric(encoded["outcome"], errors="coerce")
        row = {"model": label, "cohort_n": len(frame), "n": len(outcome),
               "dropped_numeric": len(frame) - len(outcome), "deaths": int(outcome.eq(1).sum()),
               "invalid_outcome": int((~outcome.isin([0, 1])).sum()),
               "OR": np.nan, "CI_lower": np.nan, "CI_upper": np.nan, "p_value": np.nan,
               "converged": False, "inference_valid": False, "iterations": np.nan,
               "warnings": "", "fit_error": ""}
        if row["invalid_outcome"]:
            row["fit_error"] = "InvalidOutcome"
            return None, row
        outcome = outcome.astype(int)
        fitted = None
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            try:
                design = sm.add_constant(features, has_constant="add")
                # Same unpenalized Logit/default optimizer and iteration limit as before.
                fitted = sm.Logit(outcome, design).fit(disp=False)
                interval = fitted.conf_int().loc["exposure"]
                row.update(OR=float(np.exp(fitted.params["exposure"])),
                           CI_lower=float(np.exp(interval.iloc[0])),
                           CI_upper=float(np.exp(interval.iloc[1])),
                           p_value=float(fitted.pvalues["exposure"]),
                           converged=bool(fitted.mle_retvals.get("converged", False)),
                           iterations=fitted.mle_retvals.get("iterations", np.nan))
                row["inference_valid"] = bool(
                    row["converged"] and np.isfinite([row["OR"], row["CI_lower"], row["CI_upper"]]).all()
                    and row["CI_lower"] > 0 and np.isfinite(row["p_value"]))
            except (ValueError, np.linalg.LinAlgError, ZeroDivisionError, PerfectSeparationError) as exc:
                # Do not expose row-level inputs or silently change the estimator.
                row["fit_error"] = type(exc).__name__
            warning_names = {type(w.message).__name__ for w in caught}
            row["warnings"] = ", ".join(sorted(warning_names))
            if warning_names & {"PerfectSeparationWarning", "HessianInversionWarning", "ConvergenceWarning"}:
                row["inference_valid"] = False
        return fitted, row


    df_lm_original = df.loc[original_lm_mask].copy()
    df_lm = df.loc[corrected_lm_mask].copy()
    # For included deaths, the classifier has confirmed landmark < death <= discharge.
    # The original 0-<24h prescription exposure and hospital expiration flag are retained.
    for frame in (df_lm_original, df_lm):
        frame["outcome"] = pd.to_numeric(frame["hospital_expire_flag"], errors="coerce")
        frame["exposure"] = frame["raas_any_early"].astype(int)

    landmark_fits = {}
    comparison_rows = []
    for frame, label in [(df_lm_original, "Original condition"), (df_lm, "24h landmark")]:
        for with_proxy in (False, True):
            name = label + (" + proxy" if with_proxy else "")
            fitted, row = fit_landmark_model(frame, name, with_proxy)
            landmark_fits[name] = fitted
            comparison_rows.append(row)
    landmark_model_comparison = pd.DataFrame(comparison_rows)
    # Counts are reported separately even if the two specifications use identical admissions.
    display(landmark_model_comparison)
    if not landmark_model_comparison["inference_valid"].all():
        print("At least one fit has failed/unconverged or non-finite inference; inspect status before interpreting ORs.")

    model_lm = landmark_fits["24h landmark"]
    model_lm_proxy = landmark_fits["24h landmark + proxy"]
    corrected_comparison = landmark_model_comparison.set_index("model").loc[
        ["24h landmark", "24h landmark + proxy"]].reset_index()
    primary_row = or_ci.loc[["exposure"]].assign(
        model="Primary", n=int(model.nobs), deaths=int(model.model.endog.sum()),
        converged=bool(model.mle_retvals.get("converged", False)))
    primary_row["inference_valid"] = primary_row["converged"] & np.isfinite(
        primary_row[["OR", "CI_lower", "CI_upper"]]).all(axis=1) & primary_row["CI_lower"].gt(0)
    compare_raw = pd.concat([primary_row, corrected_comparison], ignore_index=True)
    compare_or = compare_raw[["model", "n", "deaths", "converged", "inference_valid"]].copy()
    compare_or["CI95"] = compare_raw.apply(
        lambda r: f"{r['OR']:.2f} [{r['CI_lower']:.2f}, {r['CI_upper']:.2f}]", axis=1)
    compare_or["p_value"] = compare_raw["p_value"]
    _notebook_result_value = (compare_or)


**Portfolio Figure Export: Landmark Sensitivity Summary**

Export only when all displayed primary/corrected fits pass convergence and finite-inference checks. The 2026-11-17 real-data fits did not pass; the existing image is historical and is left unchanged. A skipped export does not prevent the subsequent prediction/bootstrap sections from running.


In [ ]:
with notebook_results.diagnostic('Figure 04b.D2', globals()):
    # Keep historical assets untouched when model inference has not been validated.
    if not compare_raw["inference_valid"].all():
        print("Landmark figure not refreshed: inspect nonconvergence/intervals; saved image remains historical.")
    else:
        # Export portfolio figure: landmark sensitivity summary
        assets_dir = project_root / "assets"
        assets_dir.mkdir(parents=True, exist_ok=True)
        figure_path = assets_dir / "landmark_sensitivity_summary.png"

        plot_df = compare_raw.set_index("model").loc[["Primary", "24h landmark", "24h landmark + proxy"]].reset_index()

        fig, ax = plt.subplots(figsize=(7.5, 3.4))
        y = np.arange(len(plot_df))
        ax.errorbar(
            plot_df["OR"],
            y,
            xerr=[plot_df["OR"] - plot_df["CI_lower"], plot_df["CI_upper"] - plot_df["OR"]],
            fmt="o",
            color="#2563eb",
            ecolor="#2563eb",
            elinewidth=2.2,
            capsize=5,
        )
        ax.axvline(1.0, color="#64748b", linestyle="--", linewidth=1.3)
        ax.set_xscale("log")
        ax.set_xlim(0.2, 1.2)
        ax.set_yticks(y)
        ax.set_yticklabels(plot_df["model"])
        ax.invert_yaxis()
        ax.set_xlabel("Exposure odds ratio for in-hospital mortality (log scale)")
        ax.set_title("Sensitivity analysis: early RAAS exposure")
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.tick_params(axis="y", length=0)
        fig.tight_layout()
        fig.savefig(figure_path, dpi=200, bbox_inches="tight")
        plt.show()

        figure_path.relative_to(project_root)


### 5.5.1 Supplementary Firth analysis for separation

The unpenalized primary and original/corrected sensitivity fits above are retained unchanged. An admission-type level with only live discharges creates quasi-complete separation: a parameter direction improves its predictions without changing other admissions, so increasing the iteration limit does not supply a finite ordinary MLE.

The supplementary fits below retain all admissions, exposure/outcome definitions, adjustment variables and categorical coding, and add Firth's Jeffreys-information penalty. This changes the estimator and is explicitly labeled **Firth sensitivity analysis**, rather than relabeling the original primary fit as converged. Age is centered at 60 and divided by 10 for numerical conditioning; this invertible transformation leaves the exposure contrast unchanged. Identical design/outcome rows are frequency-compressed exactly in local memory, without subsampling or exporting patient records.

The reported 95% intervals use penalized profile likelihood. Each fit checks full design rank, optimizer convergence, both profile bounds and an independently computed adjusted score. Read convergence, iterations and independent score checks in the generated table for the current execution. A separation certificate was found for each original unpenalized design. These numerical checks do not remove residual confounding. The profile intervals assume independent admissions; the added patient-cluster intervals below address repeated-admission dependence separately. The existing separate prediction/risk model is not replaced by Firth probabilities.

See [Firth validation](../docs/FIRTH_SENSITIVITY_VALIDATION.md) and the [logistf method documentation](https://cran.r-project.org/web/packages/logistf/logistf.pdf).


In [ ]:
with notebook_results.capture('Table 04b.10', globals()):
    # Explicit supplementary estimator; preserve all original MLE and risk-model objects.
    import sys
    sys.path.insert(0, str(project_root / "scripts"))
    from firth_sensitivity import fit_firth, export_firth_figure, NUMERIC, CATEGORIES
    assert num_vars == NUMERIC and cat_vars == CATEGORIES
    firth_rows = []
    firth_models = {}
    for firth_frame, firth_label, firth_proxy in [
        (df, "Primary cohort", False),
        (df_lm_original, "Original condition", False),
        (df_lm_original, "Original condition + proxy", True),
        (df_lm, "24h landmark", False),
        (df_lm, "24h landmark + proxy", True),
    ]:
        firth_fit, firth_row = fit_firth(firth_frame, firth_label, firth_proxy)
        firth_models[firth_label] = firth_fit
        firth_rows.append(firth_row)
    firth_compare_raw = pd.DataFrame(firth_rows)
    firth_output_path = project_root / "python" / "outputs" / "firth_model_comparison.csv"
    firth_output_path.parent.mkdir(parents=True, exist_ok=True)
    firth_compare_raw.to_csv(firth_output_path, index=False)  # aggregate model outputs only
    _notebook_result_value = (display(firth_compare_raw))


**Firth figure generation**

The next cell exports the existing profile figure and presents a fresh in-memory plot from its aggregate fitted rows. The patient-cluster figure is generated separately from accepted patient resamples.


In [ ]:
with notebook_results.capture('Figure 04b.1', globals()):
    # A separate path prevents replacement of the historical ordinary-MLE figure.
    firth_figure_path = export_firth_figure(
        firth_compare_raw, project_root / "assets" / "firth_landmark_sensitivity.png")
    if firth_figure_path is not None:
        display(firth_figure_path.relative_to(project_root))


**Ordinary-MLE interpretation boundary**

The saved historical ordinary-MLE execution had separation and did not converge. Its numerical output remains in the [diagnostic record](../python/outputs/mle_diagnostics.csv). Current ORs and fit status are generated from the fitted objects above. Do not interpret a nominal interval or p-value as completed inference when convergence or finite-inference checks fail. Supplementary Firth estimates and patient-cluster intervals are separate.


### 5.6 Age-Specific Adjusted Predicted Mortality Risk by Early RAAS Exposure

In [ ]:
with notebook_results.capture('Figure 04b.2', globals()):
    # Age range for prediction (adjust as needed)
    age_grid = np.arange(18, 95)

    # Fix other covariates at their mean values
    # (For binary variables, the mean represents the proportion)
    base = X.mean(axis=0)

    def make_pred_matrix(exposure_value: int) -> pd.DataFrame:
        # Create a prediction matrix by repeating the mean covariate values
        Xp = pd.DataFrame(np.tile(base.values, (len(age_grid), 1)), columns=base.index)
        # Vary age across the specified grid
        Xp["age"] = age_grid
        # Set RAAS exposure status (0 = no early RAAS, 1 = early RAAS)
        Xp["exposure"] = exposure_value
        # Ensure consistent numeric types for sklearn
        return Xp.astype(np.float64)

    # Prediction under no early RAAS exposure
    Xp0 = make_pred_matrix(0)

    # Prediction under early RAAS exposure
    Xp1 = make_pred_matrix(1)

    # Predicted mortality probabilities from the unweighted full-cohort risk model
    p0 = risk_model.predict_proba(Xp0)[:, 1]
    p1 = risk_model.predict_proba(Xp1)[:, 1]

    # Plot adjusted predicted mortality by age and RAAS exposure
    assets_dir = project_root / "assets"
    assets_dir.mkdir(parents=True, exist_ok=True)
    figure_path = assets_dir / "predicted_mortality_by_age.png"

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(age_grid, p0 * 100, label="No documented early prescription exposure", color="#64748b", linewidth=2.2)
    ax.plot(age_grid, p1 * 100, label="Documented early prescription exposure", color="#2563eb", linewidth=2.2)
    ax.set_xlabel("Age")
    ax.set_ylabel("Adjusted predicted mortality (%)")
    ax.set_title("Adjusted predicted mortality by age\nUnweighted full-cohort risk model")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend()
    fig.text(
        0.01,
        0.01,
        "Documented inpatient prescription exposure; observational model-predicted risks.",
        fontsize=9,
        color="#475569",
    )
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()

    _notebook_result_value = (figure_path.relative_to(project_root))


### 5.7 Bootstrap-Based Estimation of Age-Specific Risk Differences

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# ------------------------------
# 1) Build X, y (OHE)
# ------------------------------
def build_Xy_ohe(df, cat_vars, num_vars):
    df = df.copy()

    # Fill missing categorical values before OHE
    for c in cat_vars:
        df[c] = df[c].fillna("Unknown")

    # Define outcome/exposure as int
    df["outcome"]  = df["outcome"].astype(int)
    df["exposure"] = df["exposure"].astype(int)

    # One-hot encode categorical covariates
    df_ohe = pd.get_dummies(df, columns=cat_vars, drop_first=True)

    # Drop rows with missing numeric covariates (simple rule)
    df_ohe = df_ohe.dropna(subset=num_vars)

    # Build feature columns
    feature_cols = ["exposure"] + num_vars
    for c in cat_vars:
        feature_cols += [col for col in df_ohe.columns if col.startswith(f"{c}_")]

    X = df_ohe[feature_cols].astype(float)
    y = df_ohe["outcome"].astype(int)
    return X, y

# ------------------------------
# 2) Fit the same unweighted risk-model specification used above
# ------------------------------
def fit_risk_model(X, y):
    pipe = make_unweighted_risk_model()
    pipe.fit(X, y)
    return pipe

# ------------------------------
# 3) Utility: which column is class=1?
# ------------------------------
def death_col(model):
    classes = model.named_steps["logit"].classes_
    return int(np.where(classes == 1)[0][0])

# ------------------------------
# 4) AME (risk difference)
# ------------------------------
def compute_ame(model, X):
    dc = death_col(model)

    X0 = X.copy()
    X1 = X.copy()
    X0["exposure"] = 0
    X1["exposure"] = 1

    p0 = model.predict_proba(X0)[:, dc]
    p1 = model.predict_proba(X1)[:, dc]
    return float((p1 - p0).mean()), float(p0.mean()), float(p1.mean())

# ------------------------------
# 5) RD(age): typical-patient approach (mean covariates)
#    IMPORTANT: Use the same function for BOTH line and bootstrap.
# ------------------------------
def compute_rd_by_age(model, X, age_grid):
    dc = death_col(model)

    base = X.mean(axis=0)

    def make_X(exposure_value):
        Xp = pd.DataFrame(np.tile(base.values, (len(age_grid), 1)), columns=base.index)
        Xp["exposure"] = exposure_value
        Xp["age"] = age_grid.astype(float)
        return Xp.astype(float)

    Xp0 = make_X(0)
    Xp1 = make_X(1)

    p0 = model.predict_proba(Xp0)[:, dc]
    p1 = model.predict_proba(Xp1)[:, dc]
    return (p1 - p0).astype(float)  # RD(age)

# ------------------------------
# 6) Bootstrap for RD(age)
# ------------------------------
def bootstrap_rd_by_age(df, cat_vars, num_vars, age_grid, n_boot=200, seed=42):
    rng = np.random.default_rng(seed)
    n = len(df)

    rd_list = []

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        df_b = df.iloc[idx].copy()

        X_b, y_b = build_Xy_ohe(df_b, cat_vars, num_vars)
        if y_b.nunique() < 2:
            continue

        risk_model_b = fit_risk_model(X_b, y_b)
        rd_b = compute_rd_by_age(risk_model_b, X_b, age_grid)
        rd_list.append(rd_b)

        if (b + 1) % 25 == 0:
            print(f"bootstrap {b+1}/{n_boot}")

    return np.vstack(rd_list)  # (n_success, len(age_grid))

# ------------------------------
# 7) Run + report + plot
# ------------------------------
# Rebuild X, y and verify exact alignment with the full-cohort risk-model data
risk_X, risk_y = build_Xy_ohe(df, cat_vars, num_vars)
if list(risk_X.columns) != list(X.columns) or not risk_X.index.equals(X.index):
    raise ValueError("Risk-model design matrix does not match the earlier full-cohort fit.")
X, y = risk_X, risk_y
print(f"Rows used: {len(y):,} / {len(df):,}")
print(f"Outcome rate: {y.mean():.4%}")
print(f"Exposure rate: {X['exposure'].mean():.4%}")

# risk_model was fit once above on this same full-cohort design matrix.
print("Risk-model alignment check: passed")


### 5.8 Adjusted Average Predicted Mortality Risk by RAAS Exposure Status

In [ ]:
with notebook_results.capture('Table 04b.11', globals()):
    ame, p0_mean, p1_mean = compute_ame(risk_model, X)
    print("\nAverage predicted mortality risk:")
    print(f"  No documented early prescription exposure : {p0_mean:.4%}")
    _notebook_result_value = (print(f"  Documented early prescription exposure    : {p1_mean:.4%}"))


### 5.9 Average Marginal Effect of Early RAAS Exposure on In-Hospital Mortality

In [ ]:
with notebook_results.capture('Table 04b.12', globals()):
    print("\nAverage marginal effect (AME; risk difference):")
    _notebook_result_value = (print(f"  RD = {ame:.4%} ({ame*100:.3f} percentage points)"))


### 5.10 Age-Specific Adjusted Risk Difference of In-Hospital Mortality by Early RAAS Exposure

In [ ]:
with notebook_results.capture('Figure 04b.3', globals()):
    # RD(age) line
    age_grid = np.arange(18, 95)
    rd_age = compute_rd_by_age(risk_model, X, age_grid)

    # bootstrap CI band
    rd_boot = bootstrap_rd_by_age(df, cat_vars, num_vars, age_grid, n_boot=200, seed=42)
    rd_mean = rd_boot.mean(axis=0)
    rd_lo = np.percentile(rd_boot, 2.5, axis=0)
    rd_hi = np.percentile(rd_boot, 97.5, axis=0)

    # Point line uses the same full-cohort unweighted risk model as the other
    # adjusted-risk estimates; the band uses refits of that same specification.
    rd_line = rd_age

    # plot in percentage points and save optional README-ready asset
    assets_dir = project_root / "assets"
    assets_dir.mkdir(parents=True, exist_ok=True)
    figure_path = assets_dir / "absolute_risk_difference_plot.png"

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(age_grid, rd_line * 100, label="Exposure − no exposure", color="#2563eb")
    ax.fill_between(age_grid, rd_lo * 100, rd_hi * 100, alpha=0.25, label="95% CI", color="#93c5fd")
    ax.axhline(0, linestyle="--", color="#64748b", linewidth=1.4)
    ax.set_title("Age-specific difference in adjusted predicted mortality risk")
    ax.set_xlabel("Age")
    ax.set_ylabel("Risk difference (percentage points)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend()
    fig.text(
        0.01,
        0.01,
        "Unweighted full-cohort risk model; observational association only.",
        fontsize=9,
        color="#475569",
    )
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()

    _notebook_result_value = (figure_path.relative_to(project_root))


### 5.11 Patient-aware inference and discrimination validation

Historical admission-independent profile intervals and admission-row bootstrap results remain available above. This additional analysis resamples **patients with all their eligible admissions** within each cohort, refits the existing Firth/Jeffreys estimator, and reports 95% percentile intervals from 500 prespecified replicates. Ordinary/proxy models share draws within the same cohort. Full-data OR points, exposure, cohort definitions and adjustment variables are unchanged.

Within a resample, categorical levels that are not drawn are absent from its encoding, as in an explicitly expanded row bootstrap; no observed level is pooled or removed. A synthetic expanded-data comparison verifies this behavior. Numerical fallback, when needed, optimizes the same Jeffreys-penalized objective and must pass an independent adjusted-score threshold of 1e-5. Recovered and failed fits are counted. Intervals are withheld if fewer than 95% of the prespecified fits pass; failed fits are not silently replaced by new draws. Bootstrap fitting uses at most 300 Newton iterations, gtol 1e-6 and xtol 1e-7; these settings do not change the estimator.

The separate unweighted L2 risk model is refitted on the same primary-cohort patient draws. AME and average predicted risks are averaged over each resampled admission distribution. Age-specific contrasts fix other covariates at the resampled mean profile; they remain distinct from marginal standardization and are not an interaction test. Both weighting stages (scaler and logistic fit) use patient frequencies, checked against explicit row replication.

The patient-disjoint holdout is an internal evaluation of retrospective discrimination on the full admission cohort. It uses prescription records from 0–<24 hours while including deaths within 24 hours, and applies no 24-hour survival/continued-hospitalization eligibility filter. There is no fixed clinical prediction time or prospective prediction horizon: The generated retrospective ROC AUC and PR AUC do not describe admission-time or 24-hour prognostic performance. Patient separation and train-only preprocessing address patient overlap and preprocessing leakage, not temporal feature availability. The 24-hour landmark association analyses separately require confirmed survival and continued hospitalization strictly beyond 24 hours and evaluate subsequent in-hospital death; they use a different eligible cohort.

Retrospective discrimination is assessed with a patient-disjoint 80/20 patient split (seed 42). A newly recomputed admission-random split on deterministic subject/admission order is shown only as a reference; it is not the historical row-order-dependent AUC. Training transformations are fitted only on training data. Patients are never included as model features or exported in aggregate outputs.

Run the following cells selectively: 500 resamples refit five Firth specifications plus the risk model and may take several minutes. See [patient-aware validation](../docs/PATIENT_AWARE_VALIDATION.md).

All numerical comparisons below are generated from this execution’s fitted objects and accepted resamples. No hand-written summary table is used.


In [ ]:
with notebook_results.capture(['Table 04b.13', 'Table 04b.14', 'Table 04b.15', 'Figure 04b.4', 'Figure 04b.5'], globals()):
    # Add patient-cluster uncertainty and patient-disjoint discrimination; retain legacy outputs.
    from patient_validation import evaluate_patient_split, cluster_bootstrap, export_patient_figures
    patient_split_validation = evaluate_patient_split(df, seed=42)
    display(patient_split_validation)
    patient_scopes = {
        "Primary cohort": (df, False),
        "Original condition": (df_lm_original, False),
        "Original condition + proxy": (df_lm_original, True),
        "24h landmark": (df_lm, False),
        "24h landmark + proxy": (df_lm, True),
    }
    def patient_bootstrap_progress(completed, coefficients, failures, risk_successful, risk_failures):
        if completed % 50 == 0:
            print(f"Patient bootstrap {completed}/500; accepted Firth fits: "
              f"{ {name: len(values) for name, values in coefficients.items()} }; "
              f"risk fits: {risk_successful}")
    firth_patient_ci, risk_patient_ci, age_patient_ci = cluster_bootstrap(
        patient_scopes, n_boot=500, seed=42, progress=patient_bootstrap_progress
    )
    patient_output_dir = project_root / "python" / "outputs"
    patient_output_dir.mkdir(parents=True, exist_ok=True)
    for patient_name, patient_table in [
        ("patient_split_validation", patient_split_validation),
        ("firth_patient_bootstrap", firth_patient_ci),
        ("risk_patient_bootstrap", risk_patient_ci),
        ("age_risk_patient_bootstrap", age_patient_ci),
    ]:
        patient_table.to_csv(patient_output_dir / f"{patient_name}.csv", index=False)
    display(firth_patient_ci)
    display(risk_patient_ci)
    _notebook_result_value = (export_patient_figures(firth_compare_raw, firth_patient_ci, age_patient_ci, project_root / "assets"))


## Interpretation boundary

Use the generated result summaries and fit-status diagnostics above. These are observational prescription-record contrasts, not causal treatment effects. Repeated admissions, estimator-specific uncertainty and fixed-time prediction cohorts must be distinguished. Whole-admission LOS is descriptive. Age-specific mean-profile curves are not an exposure-by-age interaction test.

See the [historical clinical execution evidence](../docs/CLINICAL_REEXECUTION_615de01.md) and [reproducible scope](../docs/REPRODUCIBLE_SCOPE.md). The separate 04c prediction-time analysis retains later ICU transfers and must not inherit the retrospective-discrimination AUCs.
